This is a simple calculator agent which can perform basic arithmetic operations such as addition, subtraction, multiplication, and division.

The agent is built using the create_agent() method, so the tool call is abstracted.The agent is also making parallel tool calls, hence the architecture is not appropriate for performing BODMAS operation.

In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [2]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [3]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from typing import Annotated
from operator import add

In [4]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [5]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add] = Field(default_factory=list)
    result_in_figure: float | None = None
    result_with_description: str | None = None

In [6]:
@tool
def add(numbers: list[float]) -> float:
    """THis tool sum up the value of all the arguments passed into it."""
    total = 0
    # Loop through the tuple of arguments
    for num in numbers:
        total += num
    return total

In [7]:
@tool
def substract(num1:float, num2:float)->float:
    """ This tool is used to find the difference between the two numbers"""
    return num1-num2

In [8]:
@tool
def multiply(numbers: list[float]) -> float:
    """ This tool is used to multiply the values of all the arguments passed into it"""
    product=1
    # Loop through multiple arguments
    for num in numbers:
        product = product*num
    return product

In [9]:
@tool
def divide(num1:float, num2:float)->float:
    """THis tool is used to divide the argument passed into it."""
    try:
        if(num2!=0):
            return num1/num2
    except ZeroDivisionError:
        print("Can not divide by 0")

In [10]:
@tool
def save_final_result(
    result_in_figure: float,
    result_with_description: str,
    runtime: ToolRuntime,
) -> Command:
    """Save the final calculated value and its user-friendly explanation."""
    return Command(
        update={
            "result_in_figure": result_in_figure,
            "result_with_description": result_with_description,
            "messages": [
                ToolMessage(
                    content="Final result saved.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [11]:
tools = [add, substract, multiply, divide, save_final_result]

In [12]:
arithmetic_agent= create_agent(
    model= model,
    system_prompt= """
        You are a calculator assistant. Use arithmetic tools to calculate the answer.
        After calculating, call save_final_result with:
        - result_in_figure: the numeric answer
        - result_with_description: a concise explanation for the user
    """,
    tools=tools,
    state_schema= State
)

In [14]:
result = arithmetic_agent.invoke({
    "messages": [HumanMessage(content="What will be the value of $ 25 - \bigl[20 + \bigl\{12 - (5 + 3 \text{ of } 2) \div 2\bigr\} + 3^{2}\bigr] \times 2 $")]
})

print(result["result_in_figure"])
print(result["result_with_description"])

<>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
C:\Users\Asus\AppData\Local\Temp\ipykernel_26264\374695014.py:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  "messages": [HumanMessage(content="What will be the value of $ 25 - \bigl[20 + \bigl\{12 - (5 + 3 \text{ of } 2) \div 2\bigr\} + 3^{2}\bigr] \times 2 $")]


-26.0

The value of the expression 25 - [20 + {12 - (5 + 3 of 2) ÷ 2} + 3^2] × 2 is -26.